In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpi4py import MPI

import gmsh
import pyvista
from pyvista.core.errors import PyVistaDeprecationWarning

import dolfinx
from dolfinx import default_scalar_type, mesh
from dolfinx.io import gmsh as gmshio
from dolfinx.fem import (Constant, Expression, Function, functionspace,
                         locate_dofs_topological, locate_dofs_geometrical,
                         dirichletbc, form, assemble_scalar)
from dolfinx.fem.petsc import LinearProblem
from dolfinx.plot import vtk_mesh
import basix.ufl
from ufl import (SpatialCoordinate, TestFunction, TrialFunction, Measure, FacetNormal,
                 inner, dot, grad, sym, tr, sqrt, Identity, nabla_div, ds, dx)

warnings.filterwarnings("ignore", category=PyVistaDeprecationWarning, module="pyvista")
print(dolfinx.__version__)

0.12.0.dev0


In [2]:
gmsh.initialize()
gmsh.model.occ.importShapes("truss_profile_2d.brep")
gmsh.model.occ.dilate(gmsh.model.occ.getEntities(), 0,0,0, 1e-3,1e-3,1e-3)
gmsh.model.occ.synchronize()
p1 = gmsh.model.occ.addPoint(0.05, -0.143, 0)
p2 = gmsh.model.occ.addPoint(7.95, -0.143, 0)
gmsh.model.occ.fragment([(2, 1)], [(0, p1), (0, p2)])
gmsh.model.occ.synchronize()
gmsh.model.addPhysicalGroup(2, [1], 1)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)   # 50 mm elements
gmsh.model.mesh.generate(2)
gmsh.model.mesh.setOrder(2)

Info    : Meshing 1D...                                                                                                                         
Info    : [  0%] Meshing curve 1 (TrimmedCurve)
Info    : [ 10%] Meshing curve 2 (TrimmedCurve)
Info    : [ 10%] Meshing curve 3 (TrimmedCurve)
Info    : [ 10%] Meshing curve 4 (TrimmedCurve)
Info    : [ 20%] Meshing curve 5 (TrimmedCurve)
Info    : [ 20%] Meshing curve 6 (TrimmedCurve)
Info    : [ 20%] Meshing curve 7 (TrimmedCurve)
Info    : [ 20%] Meshing curve 8 (TrimmedCurve)
Info    : [ 30%] Meshing curve 9 (TrimmedCurve)
Info    : [ 30%] Meshing curve 10 (TrimmedCurve)
Info    : [ 30%] Meshing curve 11 (TrimmedCurve)
Info    : [ 40%] Meshing curve 12 (TrimmedCurve)
Info    : [ 40%] Meshing curve 13 (TrimmedCurve)
Info    : [ 40%] Meshing curve 14 (TrimmedCurve)
Info    : [ 40%] Meshing curve 15 (TrimmedCurve)
Info    : [ 50%] Meshing curve 16 (TrimmedCurve)
Info    : [ 50%] Meshing curve 17 (TrimmedCurve)
Info    : [ 50%] Meshing curve 

In [5]:
mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, 0, gdim=2)
domain = mesh_data.mesh
gmsh.finalize()

In [6]:
fdim = domain.topology.dim - 1                       # edges = 1
domain.topology.create_connectivity(fdim, domain.topology.dim)

y_bot = -0.143
def left_support(x):
    return np.isclose(x[1], y_bot) & (x[0] <= 0.05 + 1e-9)

def right_support(x):
    return np.isclose(x[1], y_bot) & (x[0] >= 7.95 - 1e-9)

def roof(x):                                         # outer top edge
    centreline = (4/3) * (1 - np.abs(x[0] - 4.0) / 4.0)
    return x[1] > centreline + 0.10

f_left  = mesh.locate_entities_boundary(domain, fdim, left_support)
f_right = mesh.locate_entities_boundary(domain, fdim, right_support)
f_roof  = mesh.locate_entities_boundary(domain, fdim, roof)

# split the roof at the ridge using each edge's midpoint
mid = mesh.compute_midpoints(domain, fdim, f_roof)
f_roofL = f_roof[mid[:, 0] < 4.0]
f_roofR = f_roof[mid[:, 0] >= 4.0]

indices = np.hstack([f_left, f_right, f_roofL, f_roofR]).astype(np.int32)
values  = np.hstack([np.full(len(f_left), 11), np.full(len(f_right), 12),
                     np.full(len(f_roofL), 21), np.full(len(f_roofR), 22)]).astype(np.int32)
order = np.argsort(indices)
facet_tags = mesh.meshtags(domain, fdim, indices[order], values[order])